# Notebook 06 — Full Model and Leave-One-Out Ablation Matrix

**Summary:** Closes the loop — assembles the full proposed model and runs the complete
leave-one-out ablation sweep. Produces the master results table for the thesis.

**Full model components:**
- C1: R-Blur foveal transform (`src/foveation.py`)
- C2: Trace-based metameric periphery (Watson-SNR grounded)
- C3: VOneBlock Poisson noise (`src/overrides.py`)
- C4: ResNet-50 ventral back-end (`src/models.py`)
- C5: Multi-glance + confidence halting (`src/it_feedback.py`)

**Outputs:** `results/06_ablation_table.json` · `results/06_ablation_table.png` ·
`results/06_efficiency_curve.png`

---

## Mathematics

**Marginal contribution** of component $C_k$:

$$
\Delta_k = \mathrm{metric}(\text{full model}) - \mathrm{metric}(\text{full} \setminus C_k)
$$

**EOT adversarial gradient** (for stochastic C3):

$$
\bar g = \frac{1}{N}\sum_{i=1}^N \nabla_x \ell(f(T_i(x)),\, y), \quad N\ge10
$$

**Efficiency curve:** accuracy $A$ vs average glances $G$:

$$
\eta = \frac{A}{G} \quad \text{(accuracy-per-glance efficiency)}
$$

> **Smoke-test scope:** All models use random weights; accuracy numbers are not
> meaningful. The notebook demonstrates the full pipeline and ablation machinery.


In [ ]:
# ── Environment ─────────────────────────────────────────────────────────────
import sys, os, warnings, json
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/tez_foveated_vision"
else:
    _here = os.getcwd()
    for _d in [_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))]:
        if os.path.isdir(os.path.join(_d, "src")):
            PROJECT_ROOT = _d; break
    else: PROJECT_ROOT = _here

os.chdir(PROJECT_ROOT)
for _p in [PROJECT_ROOT,
           os.path.join(PROJECT_ROOT, "external", "vonenet"),
           os.path.join(PROJECT_ROOT, "external", "CORnet")]:
    if _p not in sys.path: sys.path.insert(0, _p)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src import common, models
from src.foveation import (SNRProfile, build_foveated_transform,
                            apply_rblur, FoveatedTransform, TraceBasedPeriphery)
from src.overrides  import apply_vone_block_input_gradient_fix, set_v1_noise_mode
from src.it_feedback import FixationGrid, MultiGlanceFoveatedModel
from src.eval_harness import pgd_attack

CFG = {
    "seed": 42, "smoke_test": True,
    "image_size": 32, "n_classes": 10, "ppd": 4.0,
    "fovea_deg": 1.0, "transition_deg": 0.5,
    "rblur_sigma0": 0.5, "rblur_slope": 1.5,
    "snr0_db": 30.0, "beta": 2.0, "patch_size": 8,
    "max_glances": 4, "halt_threshold": 0.75, "margin": 0.25,
    # Training
    "n_epochs_smoke": 3, "n_batches_smoke": 20,
    "batch_size": 32, "lr": 1e-3,
    # Adversarial
    "eps": 4/255, "pgd_steps": 5, "pgd_alpha": 1/255, "eot_samples": 4,
    "result_dir": os.path.join(PROJECT_ROOT, "results"),
    "ckpt_dir":   os.path.join(PROJECT_ROOT, "checkpoints"),
}

common.set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CFG["result_dir"], exist_ok=True)
apply_vone_block_input_gradient_fix()
print(f"Device: {device} | Smoke test: {CFG['smoke_test']}")


## Assembling the Full Model

The full model pipeline:

```
x_raw  (raw [0,1] pixels)
  ↓  FoveatedTransform (C1: R-Blur  +  C2: trace-based periphery)
  ↓  NormalizedModel(VOneResNet-50, C3: Poisson noise, C4: ResNet-50 back-end)
  ↓  MultiGlanceFoveatedModel (C5: multi-glance + confidence halting)
  →  logits
```

For leave-one-out: each ablation removes exactly one component and keeps all others.


In [ ]:
# ── Build the full model and all ablation variants ───────────────────────

PRETRAINED = not CFG["smoke_test"]
S = CFG["image_size"]
snr = SNRProfile(snr0_db=CFG["snr0_db"], beta=CFG["beta"], ppd=CFG["ppd"])
grid = FixationGrid(2, 2, CFG["margin"])
fixations = grid.get_fixations()

MU  = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class FullPipeline(nn.Module):
    """Full foveated model: fov_transform -> backbone -> (optional multi-glance).

    Accepts raw [0,1] pixel tensors. Applies foveation then backbone normalisation.
    """
    def __init__(self, fov_transform, backbone, norm, fixations_list=None,
                 halt_threshold=0.75, sigma0=0.5, slope=1.5, ppd=4.0,
                 multi_glance=False):
        super().__init__()
        self.fov = fov_transform     # None -> no foveation
        self.backbone = models.NormalizedModel(backbone, norm)
        self.multi_glance = multi_glance
        self.fixations = fixations_list or [(0.5, 0.5)]
        self.halt_threshold = halt_threshold
        self.sigma0 = sigma0; self.slope = slope; self.ppd = ppd

    def apply_fov(self, x_raw, fixation_yx=None):
        if self.fov is None:
            return x_raw
        # Temporarily override fixation in fov transform's periphery modules
        if fixation_yx is not None and hasattr(self.fov, 'fixation_yx'):
            self.fov.fixation_yx = fixation_yx
            if self.fov.periphery is not None:
                self.fov.periphery.fixation_yx = fixation_yx
        return torch.stack([self.fov(x_raw[i]) for i in range(x_raw.size(0))])

    def forward(self, x_raw, return_glance_count=False):
        B = x_raw.size(0)
        if not self.multi_glance:
            fov_x = self.apply_fov(x_raw)
            logits = self.backbone(fov_x)
            if return_glance_count:
                return logits, torch.ones(B, dtype=torch.long)
            return logits

        logit_sum = None
        halted = torch.zeros(B, dtype=torch.bool)
        gc = torch.ones(B, dtype=torch.long)
        for t, fix in enumerate(self.fixations):
            fov_x = self.apply_fov(x_raw, fix)
            z = self.backbone(fov_x)
            logit_sum = z if logit_sum is None else logit_sum + z
            if not self.training and t < len(self.fixations) - 1:
                conf = F.softmax(logit_sum / (t+1), dim=1).max(1).values
                new_halt = (conf >= self.halt_threshold) & (~halted)
                gc += (~halted & ~new_halt).long()
                halted |= new_halt
                if halted.all(): break
        final = logit_sum / len(self.fixations)
        return (final, gc) if return_glance_count else final


def build_variant(label, PRETRAINED, snr, fixations, CFG):
    """Build one ablation variant. Returns FullPipeline."""
    # Default: full model
    use_fov    = True   # C1
    use_trace  = True   # C2
    use_vnoise = True   # C3
    use_cornet = False  # C4 variant (True = CORnet-S instead of ResNet-50)
    use_multi  = True   # C5

    if   label == "full":         pass
    elif label == "no_C1_fov":    use_fov    = False
    elif label == "no_C2_trace":  use_trace  = False
    elif label == "no_C3_noise":  use_vnoise = False
    elif label == "no_C4_resnet": use_cornet = True
    elif label == "no_C5_multi":  use_multi  = False

    # Back-end
    if use_cornet:
        backbone, norm = models.build_cornet_s(pretrained=PRETRAINED)
    else:
        backbone, norm = models.build_voneresnet50(pretrained=PRETRAINED)
        set_v1_noise_mode(backbone, mode="neuronal" if use_vnoise else None)

    backbone.eval()

    # Foveation transform
    if not use_fov:
        ft = None
    elif not use_trace:
        ft = build_foveated_transform("blur", snr, CFG["patch_size"],
                                       CFG["fovea_deg"], CFG["transition_deg"],
                                       CFG["ppd"], (0.5, 0.5),
                                       CFG["rblur_sigma0"], CFG["rblur_slope"])
    else:
        ft = build_foveated_transform("trace", snr, CFG["patch_size"],
                                       CFG["fovea_deg"], CFG["transition_deg"],
                                       CFG["ppd"], (0.5, 0.5))

    return FullPipeline(ft, backbone, norm,
                        fixations_list=fixations if use_multi else [(0.5, 0.5)],
                        halt_threshold=CFG["halt_threshold"],
                        sigma0=CFG["rblur_sigma0"],
                        slope=CFG["rblur_slope"],
                        ppd=CFG["ppd"],
                        multi_glance=use_multi)


ABLATION_LABELS = ["full", "no_C1_fov", "no_C2_trace",
                    "no_C3_noise", "no_C4_resnet", "no_C5_multi"]

print("Building ablation variants...")
ablation_models = {}
for lbl in ABLATION_LABELS:
    ablation_models[lbl] = build_variant(lbl, PRETRAINED, snr, fixations, CFG).to(device)
    n_params = sum(p.numel() for p in ablation_models[lbl].parameters()) / 1e6
    print(f"  {lbl:20s}  {n_params:.1f}M params")


In [ ]:
# ── Synthetic dataset + training / evaluation ─────────────────────────────

class SyntheticCIFAR(torch.utils.data.Dataset):
    def __init__(self, n=512, H=32, W=32, n_classes=10, seed=42):
        rng = np.random.RandomState(seed)
        self.images = torch.rand(n, 3, H, W)
        self.labels = torch.from_numpy(rng.randint(0, n_classes, n)).long()
        self.images = (self.images - MU) / STD
    def __len__(self): return len(self.images)
    def __getitem__(self, i): return self.images[i], self.labels[i]

train_ld = torch.utils.data.DataLoader(SyntheticCIFAR(512, seed=42),
                                        batch_size=CFG["batch_size"], shuffle=True)
val_ld   = torch.utils.data.DataLoader(SyntheticCIFAR(128, seed=99),
                                        batch_size=CFG["batch_size"], shuffle=False)

# Add a small linear head on top of each full pipeline for training
# (In the full run, the backbone is fine-tuned end-to-end.)
class AblationTrainer(nn.Module):
    def __init__(self, pipeline, n_classes, image_size=32):
        super().__init__()
        self.pipeline = pipeline
        # Detect output dimension from a dummy forward pass
        with torch.no_grad():
            _dummy = pipeline(torch.rand(1, 3, image_size, image_size).to(
                next(pipeline.parameters()).device))
        out_dim = _dummy.shape[-1]   # e.g. 1000 for ImageNet backbone
        self.head = nn.Linear(out_dim, n_classes)
    def forward(self, x_raw):
        logits = self.pipeline(x_raw)
        return self.head(logits)


def train_ablation(pipeline, n_classes, loader, n_batches):
    trainer = AblationTrainer(pipeline, CFG["n_classes"], CFG["image_size"]).to(device)
    opt    = torch.optim.Adam(trainer.head.parameters(), lr=CFG["lr"])
    trainer.train()
    for bi, (xb, yb) in enumerate(loader):
        if bi >= n_batches: break
        xb, yb = xb.to(device), yb.to(device)
        x_raw = (xb * STD.to(device) + MU.to(device)).clamp(0.0, 1.0)
        logits = trainer(x_raw)
        loss = F.cross_entropy(logits, yb)
        opt.zero_grad(); loss.backward(); opt.step()
    return trainer


@torch.no_grad()
def eval_pipeline(pipeline, loader, is_stochastic=False):
    pipeline.eval()
    correct = total = total_gc = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        x_raw = (xb * STD.to(device) + MU.to(device)).clamp(0.0, 1.0)
        out = pipeline(x_raw, return_glance_count=True)
        if isinstance(out, tuple):
            logits, gc = out
            total_gc += gc.sum().item()
        else:
            logits = out
            total_gc += xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total   += xb.size(0)
    return correct / max(total, 1), total_gc / max(total, 1)


In [ ]:
# ── Run leave-one-out ablation ────────────────────────────────────────────

print("Running leave-one-out ablation sweep (smoke_test)...")
print(f"{'Variant':20s}  {'Clean acc':>10}  {'Avg glances':>12}  {'Efficiency':>10}")
print("-" * 58)

ablation_results = {}
for lbl, pipeline in ablation_models.items():
    common.set_seed(CFG["seed"])
    # Quick training (head only — maps backbone output -> n_classes)
    trainer = AblationTrainer(pipeline, CFG["n_classes"], CFG["image_size"])

    # Evaluate
    pipeline.eval()
    val_acc, avg_gc = eval_pipeline(pipeline, val_ld)
    efficiency = val_acc / max(avg_gc, 1)

    ablation_results[lbl] = {
        "val_acc": val_acc,
        "avg_glances": avg_gc,
        "efficiency": efficiency,
    }
    print(f"  {lbl:20s}  {val_acc:10.4f}  {avg_gc:12.2f}  {efficiency:10.4f}")

# Compute marginal contributions relative to full model
full_acc = ablation_results["full"]["val_acc"]
print(f"\n{'Marginal contribution Delta_k = full - ablated':}")
for lbl, res in ablation_results.items():
    if lbl == "full": continue
    delta = full_acc - res["val_acc"]
    print(f"  {lbl:20s}  Delta_k = {delta:+.4f}")

print("\n(Smoke_test: random weights -> all deltas are noise. Run full training for real results.)")


In [ ]:
# ── Adversarial robustness for full model ─────────────────────────────────

print("PGD robustness evaluation (full model, EOT)...")

full_pipeline = ablation_models["full"]
full_wrapped  = full_pipeline.backbone   # NormalizedModel already inside

# Small batch from val set
xb, yb = next(iter(val_ld))
xb, yb = xb.to(device), yb.to(device)
x_raw  = (xb * STD.to(device) + MU.to(device)).clamp(0.0, 1.0)

# PGD against the full pipeline (which acts as a NormalizedModel + foveation)
# EOT-N: needed because C3 (VOneBlock Poisson) makes the forward stochastic
x_adv = pgd_attack(full_pipeline, x_raw, yb,
                    eps=CFG["eps"], alpha=CFG["pgd_alpha"],
                    steps=CFG["pgd_steps"], eot_samples=CFG["eot_samples"])

with torch.no_grad():
    clean_logits = full_pipeline(x_raw)
    adv_logits   = full_pipeline(x_adv)

clean_acc = (clean_logits.argmax(1) == yb).float().mean().item()
adv_acc   = (adv_logits.argmax(1)   == yb).float().mean().item()
print(f"  Clean accuracy  : {clean_acc:.4f}")
print(f"  Robust accuracy : {adv_acc:.4f}  (PGD eps={CFG['eps']:.4f}, EOT={CFG['eot_samples']})")
print("  (Values meaningless with random weights; meaningful after full training.)")


In [ ]:
# ── Ablation table figure ─────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels_short = [l.replace("no_", "-").replace("_", " ") for l in ABLATION_LABELS]
accs = [ablation_results[l]["val_acc"]     for l in ABLATION_LABELS]
gcs  = [ablation_results[l]["avg_glances"] for l in ABLATION_LABELS]
effs = [ablation_results[l]["efficiency"]  for l in ABLATION_LABELS]

colors = plt.cm.Set2(np.linspace(0, 1, len(ABLATION_LABELS)))

ax = axes[0]
bars = ax.barh(range(len(ABLATION_LABELS)), accs, color=colors)
ax.set_yticks(range(len(ABLATION_LABELS)))
ax.set_yticklabels(labels_short, fontsize=9)
ax.set_xlabel("Val accuracy")
ax.set_title("Leave-one-out ablation: clean accuracy\n(smoke_test — not meaningful)")
ax.axvline(accs[0], color="k", lw=1, ls="--", alpha=0.5, label="full model")
ax.legend(fontsize=8); ax.grid(axis="x", alpha=0.3)

ax = axes[1]
ax.scatter(gcs, accs, c=range(len(ABLATION_LABELS)), cmap="Set2", s=120, zorder=5)
for i, lbl in enumerate(labels_short):
    ax.annotate(lbl, (gcs[i], accs[i]), textcoords="offset points",
                xytext=(5, 3), fontsize=7)
ax.set_xlabel("Avg glances per image")
ax.set_ylabel("Val accuracy")
ax.set_title("Efficiency curve: accuracy vs glances\n(each point = one ablation condition)")
ax.grid(True, alpha=0.3)

fig.suptitle(
    "Full model leave-one-out ablation matrix\n"
    "(smoke_test=True: random weights + synthetic data — for pipeline verification only)",
    fontsize=9, y=1.02,
)
plt.tight_layout()
tbl_path = os.path.join(CFG["result_dir"], "06_ablation_table.png")
common.save_figure(fig, tbl_path, dpi=130)
plt.close(fig)
print(f"Saved: {tbl_path}")


In [ ]:
# ── Save results ─────────────────────────────────────────────────────────
summary = {
    "notebook": "06_full_model_and_ablations",
    "cfg":      {k: v for k, v in CFG.items() if not k.endswith("_dir")},
    "ablation_table": ablation_results,
    "adversarial": {
        "clean_acc": clean_acc, "robust_acc": adv_acc,
        "eps": CFG["eps"], "eot_samples": CFG["eot_samples"],
    },
    "note": "smoke_test=True: all numbers reflect pipeline, not real performance.",
}
rpath = os.path.join(CFG["result_dir"], "06_ablation_table.json")
common.save_json(summary, rpath)
common.save_config(CFG, os.path.join(CFG["result_dir"], "06_config.json"))

print(f"Results: {rpath}")
print(f"06_ablation_table.png : {os.path.exists(tbl_path)}")
print()
print("=" * 60)
print("ALL NOTEBOOKS COMPLETE")
print("=" * 60)
nbs = ["00_setup_and_data", "01_baseline_reproduce",
       "02_foveation_rblur_and_periphery", "03_v1_block",
       "04_it_feedback_multiglance", "05_mftma_certification",
       "06_full_model_and_ablations"]
for nb in nbs:
    exists = os.path.exists(os.path.join(PROJECT_ROOT, "notebooks", nb + ".ipynb"))
    print(f"  {'OK' if exists else 'MISSING':6s} notebooks/{nb}.ipynb")
print()
src_files = ["common.py", "datasets.py", "eval_harness.py", "models.py",
             "overrides.py", "foveation.py", "it_feedback.py", "mftma.py"]
for sf in src_files:
    exists = os.path.exists(os.path.join(PROJECT_ROOT, "src", sf))
    print(f"  {'OK' if exists else 'MISSING':6s} src/{sf}")
